# License Plate Recognition — Training Pipeline (PP-OCRv5 vs PP-OCRv4)

Fine-tunes PaddleOCR's **text recognition** models — `PP-OCRv5_server_rec` and
`PP-OCRv4_server_rec` — on a real, self-collected license-plate dataset (200 car photos with
ground-truth plate numbers checked into this repo's `images/` folder), then evaluates, exports
and publishes both models as a GitHub Release so the inference notebook can download them.

**Everything needed lives in this repo** — nothing to upload, no Google Drive. You will need a
GitHub [personal access token](https://github.com/settings/tokens) (classic, `repo` scope) at
the end to publish the trained models. Use a GPU runtime: `Runtime > Change runtime type > T4
GPU`.

## 1. Setup

In [ ]:
%cd /content
!git clone -q https://github.com/parushurambannela/license-plate-recognition-paddleocr.git
%cd license-plate-recognition-paddleocr

import os

IMAGES_DIR = "/content/license-plate-recognition-paddleocr/images"
ARTIFACTS_DIR = "/content/artifacts"
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

In [ ]:
# Colab's default GPU runtime uses CUDA 12.x - if this install fails, check your CUDA
# version with `!nvidia-smi` and pick a matching tag from paddlepaddle.org.cn/install/quick
!pip install -q paddlepaddle-gpu -i https://www.paddlepaddle.org.cn/packages/stable/cu121/
!pip install -q paddleocr

In [ ]:
%cd /content
!rm -rf PaddleOCR
!git clone -q --depth 1 https://github.com/PaddlePaddle/PaddleOCR.git
%cd PaddleOCR
!pip install -q -r requirements.txt

PADDLEOCR_REPO = "/content/PaddleOCR"
print("Ready:", PADDLEOCR_REPO)

## 2. Dataset

200 real car photos with ground-truth plate numbers, already sitting in this repo's `images/`
folder (`images/labels.csv` maps each filename to its plate text). Since the photos are full
scenes rather than pre-cropped plates, PaddleOCR's pretrained pipeline is run once per photo to
find the plate: whichever detected text box's recognized text is closest to the known ground
truth is kept as the crop.

- 170 images -> auto-cropped -> fine-tuning set (80% train / 20% val)
- 30 images -> held out untouched (full photos) -> published for the inference notebook

In [ ]:
import csv

with open(os.path.join(IMAGES_DIR, "labels.csv")) as f:
    records = [(row["image"], row["text"]) for row in csv.DictReader(f)]

print(f"Loaded {len(records)} labeled photos")

In [ ]:
import random

random.seed(0)
random.shuffle(records)

test_pool = records[:30]
work_pool = records[30:]
print(f"Work pool (for fine-tuning): {len(work_pool)} | Held-out test pool: {len(test_pool)}")

In [ ]:
import shutil

TEST_DIR = os.path.join(ARTIFACTS_DIR, "test_images")
os.makedirs(TEST_DIR, exist_ok=True)

with open(os.path.join(TEST_DIR, "labels.csv"), "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["image", "text"])
    for fname, text in test_pool:
        shutil.copy(os.path.join(IMAGES_DIR, fname), os.path.join(TEST_DIR, fname))
        writer.writerow([fname, text])

print("Held-out test photos staged at", TEST_DIR)

In [ ]:
import difflib
from paddleocr import PaddleOCR
from PIL import Image

localizer = PaddleOCR(use_doc_orientation_classify=False, use_doc_unwarping=False, use_textline_orientation=False)

def crop_plate(image_path, gt_text):
    res = localizer.predict(image_path)[0].json["res"]
    best_box, best_score = None, 0.3  # ignore weak matches - probably not the plate
    for text, box in zip(res.get("rec_texts", []), res.get("rec_boxes", [])):
        score = difflib.SequenceMatcher(None, text.upper(), gt_text.upper()).ratio()
        if score > best_score:
            best_score, best_box = score, box
    if best_box is None:
        return None
    x0, y0, x1, y1 = [int(v) for v in best_box]
    return Image.open(image_path).convert("RGB").crop((x0, y0, x1, y1))

In [ ]:
DATASET_DIR = os.path.join(ARTIFACTS_DIR, "dataset")
CROPPED_IMAGES_DIR = os.path.join(DATASET_DIR, "images")
os.makedirs(CROPPED_IMAGES_DIR, exist_ok=True)

cropped_records = []
for fname, text in work_pool:
    crop = crop_plate(os.path.join(IMAGES_DIR, fname), text)
    if crop is None:
        continue
    out_name = os.path.splitext(fname)[0] + ".jpg"
    crop.save(os.path.join(CROPPED_IMAGES_DIR, out_name), quality=95)
    cropped_records.append((f"images/{out_name}", text))

print(f"Auto-cropped {len(cropped_records)} / {len(work_pool)} plates")

In [ ]:
random.shuffle(cropped_records)
split = int(len(cropped_records) * 0.8)
train_records, val_records = cropped_records[:split], cropped_records[split:]

with open(os.path.join(DATASET_DIR, "train.txt"), "w") as f:
    for path, text in train_records:
        f.write(f"{path}\t{text}\n")

with open(os.path.join(DATASET_DIR, "val.txt"), "w") as f:
    for path, text in val_records:
        f.write(f"{path}\t{text}\n")

print(f"Train: {len(train_records)} | Val: {len(val_records)}")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 4, figsize=(14, 5))
for ax, (path, text) in zip(axes.flat, random.sample(cropped_records, 8)):
    ax.imshow(Image.open(os.path.join(DATASET_DIR, path)))
    ax.set_title(text, fontsize=9)
    ax.axis("off")
plt.suptitle("Auto-cropped plate images used for fine-tuning")
plt.tight_layout()
plt.show()

## 3. Download Pre-trained Models

In [ ]:
PRETRAIN_DIR = os.path.join(PADDLEOCR_REPO, "pretrain_models")
os.makedirs(PRETRAIN_DIR, exist_ok=True)

PRETRAINED_URLS = {
    "v5": "https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/PP-OCRv5_server_rec_pretrained.pdparams",
    "v4": "https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/PP-OCRv4_server_rec_pretrained.pdparams",
}

PRETRAINED_PATHS = {}
for tag, url in PRETRAINED_URLS.items():
    dest = os.path.join(PRETRAIN_DIR, os.path.basename(url))
    !wget -q {url} -O {dest}
    PRETRAINED_PATHS[tag] = dest.replace(".pdparams", "")
    print(tag, "->", dest)

## 4. Training

Each base config is loaded, the dataset paths / epoch count / batch size are edited in Python,
and the result is saved as a `*_custom.yml` next to it. `tools/train.py` is then run against
that custom config for each model — simpler and less error-prone than passing nested
list/dict overrides on the command line.

In [ ]:
import yaml

EPOCHS = 20
BATCH_SIZE = 8
LOG_DIR = os.path.join(ARTIFACTS_DIR, "logs")
os.makedirs(LOG_DIR, exist_ok=True)

def prepare_config(base_config_path, pretrained_model, output_dir):
    with open(base_config_path) as f:
        cfg = yaml.safe_load(f)

    cfg["Global"]["pretrained_model"] = pretrained_model
    cfg["Global"]["save_model_dir"] = output_dir
    cfg["Global"]["epoch_num"] = EPOCHS
    cfg["Global"]["save_epoch_step"] = 5
    cfg["Global"]["eval_batch_step"] = [0, 10]

    cfg["Train"]["dataset"]["data_dir"] = DATASET_DIR
    cfg["Train"]["dataset"]["label_file_list"] = [os.path.join(DATASET_DIR, "train.txt")]
    cfg["Train"]["sampler"]["first_bs"] = BATCH_SIZE
    cfg["Train"]["loader"]["num_workers"] = 2

    cfg["Eval"]["dataset"]["data_dir"] = DATASET_DIR
    cfg["Eval"]["dataset"]["label_file_list"] = [os.path.join(DATASET_DIR, "val.txt")]
    cfg["Eval"]["loader"]["batch_size_per_card"] = BATCH_SIZE
    cfg["Eval"]["loader"]["num_workers"] = 2

    out_path = base_config_path.replace(".yml", "_custom.yml")
    with open(out_path, "w") as f:
        yaml.safe_dump(cfg, f)
    return out_path

MODEL_SPECS = {
    "v5": {
        "base_config": os.path.join(PADDLEOCR_REPO, "configs/rec/PP-OCRv5/PP-OCRv5_server_rec.yml"),
        "pretrained": PRETRAINED_PATHS["v5"],
        "checkpoints_dir": os.path.join(ARTIFACTS_DIR, "trained_models/v5/checkpoints"),
        "inference_dir": os.path.join(ARTIFACTS_DIR, "trained_models/v5/inference"),
    },
    "v4": {
        "base_config": os.path.join(PADDLEOCR_REPO, "configs/rec/PP-OCRv4/PP-OCRv4_server_rec.yml"),
        "pretrained": PRETRAINED_PATHS["v4"],
        "checkpoints_dir": os.path.join(ARTIFACTS_DIR, "trained_models/v4/checkpoints"),
        "inference_dir": os.path.join(ARTIFACTS_DIR, "trained_models/v4/inference"),
    },
}

for tag, spec in MODEL_SPECS.items():
    spec["custom_config"] = prepare_config(spec["base_config"], spec["pretrained"], spec["checkpoints_dir"])
    print(tag, "->", spec["custom_config"])

### 4.1 Train PP-OCRv5_server_rec

In [ ]:
%cd {PADDLEOCR_REPO}
v5_log = os.path.join(LOG_DIR, "train_v5.log")
!python3 tools/train.py -c {MODEL_SPECS['v5']['custom_config']} 2>&1 | tee {v5_log}

### 4.2 Train PP-OCRv4_server_rec

In [ ]:
%cd {PADDLEOCR_REPO}
v4_log = os.path.join(LOG_DIR, "train_v4.log")
!python3 tools/train.py -c {MODEL_SPECS['v4']['custom_config']} 2>&1 | tee {v4_log}

## 5. Training Curves

In [ ]:
import re

def parse_train_log(log_path):
    loss_pattern = re.compile(r"epoch:\s*\[(\d+)/\d+\].*?loss:\s*([\d.]+)")
    acc_pattern = re.compile(r"cur metric,\s*acc:\s*([\d.]+)")
    losses, eval_accs = [], []
    with open(log_path, errors="ignore") as f:
        for line in f:
            m = loss_pattern.search(line)
            if m:
                losses.append(float(m.group(2)))
            m2 = acc_pattern.search(line)
            if m2:
                eval_accs.append(float(m2.group(1)))
    return losses, eval_accs

v5_losses, v5_eval_accs = parse_train_log(v5_log)
v4_losses, v4_eval_accs = parse_train_log(v4_log)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(v5_losses, label="PP-OCRv5")
axes[0].plot(v4_losses, label="PP-OCRv4")
axes[0].set_title("Training loss")
axes[0].legend()

axes[1].plot(v5_eval_accs, marker="o", label="PP-OCRv5")
axes[1].plot(v4_eval_accs, marker="o", label="PP-OCRv4")
axes[1].set_title("Validation accuracy per evaluation")
axes[1].legend()
plt.tight_layout()
plt.show()

## 6. Evaluation

In [ ]:
%cd {PADDLEOCR_REPO}

eval_results = {}
for tag, spec in MODEL_SPECS.items():
    best_ckpt = os.path.join(spec["checkpoints_dir"], "best_accuracy")
    eval_log = os.path.join(LOG_DIR, f"eval_{tag}.log")
    !python3 tools/eval.py -c {spec['custom_config']} -o Global.pretrained_model={best_ckpt} 2>&1 | tee {eval_log}

    acc, edit_dist = None, None
    with open(eval_log, errors="ignore") as f:
        for line in f:
            m = re.search(r"acc:\s*([\d.]+)", line)
            m2 = re.search(r"norm_edit_dis:\s*([\d.]+)", line)
            if m:
                acc = float(m.group(1))
            if m2:
                edit_dist = float(m2.group(1))
    eval_results[tag] = {"acc": acc, "norm_edit_dis": edit_dist}
    print(tag, eval_results[tag])

## 7. Export to Inference Format

In [ ]:
%cd {PADDLEOCR_REPO}

for tag, spec in MODEL_SPECS.items():
    best_ckpt = os.path.join(spec["checkpoints_dir"], "best_accuracy")
    os.makedirs(spec["inference_dir"], exist_ok=True)
    !python3 tools/export_model.py -c {spec['custom_config']} -o Global.pretrained_model={best_ckpt} Global.save_inference_dir={spec['inference_dir']} 2>&1 | tail -20
    print(tag, "->", spec["inference_dir"])

## 8. Results Comparison

In [ ]:
import pandas as pd

summary_df = pd.DataFrame([
    {"model": "PP-OCRv5_server_rec", "val_accuracy": eval_results["v5"]["acc"], "val_norm_edit_distance": eval_results["v5"]["norm_edit_dis"]},
    {"model": "PP-OCRv4_server_rec", "val_accuracy": eval_results["v4"]["acc"], "val_norm_edit_distance": eval_results["v4"]["norm_edit_dis"]},
])
summary_df

## 9. Publish Models for the Inference Notebook

Zips the two exported models plus the held-out test photos, and publishes them as assets on a
GitHub Release (tag `trained-models`) of this same repo. The inference notebook downloads them from
there with a plain `wget` — no Drive, no manual upload. Re-running this replaces the previous
release with the newly trained weights.

You'll need a GitHub [personal access token](https://github.com/settings/tokens) (classic,
`repo` scope) for `parushurambannela/license-plate-recognition-paddleocr` — pasted below, never stored in the notebook itself.

In [ ]:
import shutil

v5_zip = shutil.make_archive("/content/v5_inference", "zip", MODEL_SPECS["v5"]["inference_dir"])
v4_zip = shutil.make_archive("/content/v4_inference", "zip", MODEL_SPECS["v4"]["inference_dir"])
test_zip = shutil.make_archive("/content/test_images", "zip", TEST_DIR)
print(v5_zip, v4_zip, test_zip)

In [ ]:
import getpass
import os
import requests

token = getpass.getpass("GitHub personal access token (repo scope): ")
headers = {"Authorization": f"token {token}", "Accept": "application/vnd.github+json"}

existing = requests.get(f"https://api.github.com/repos/parushurambannela/license-plate-recognition-paddleocr/releases/tags/trained-models", headers=headers)
if existing.status_code == 200:
    requests.delete(f"https://api.github.com/repos/parushurambannela/license-plate-recognition-paddleocr/releases/{existing.json()['id']}", headers=headers)

release = requests.post(
    f"https://api.github.com/repos/parushurambannela/license-plate-recognition-paddleocr/releases",
    headers=headers,
    json={"tag_name": "trained-models", "name": "Fine-tuned models", "body": "Auto-published by the training notebook."},
).json()

upload_url = release["upload_url"].split("{")[0]

for path in [v5_zip, v4_zip, test_zip]:
    name = os.path.basename(path)
    with open(path, "rb") as f:
        r = requests.post(f"{upload_url}?name={name}", headers={**headers, "Content-Type": "application/zip"}, data=f.read())
    print(name, "->", r.status_code)

print("Release published:", release.get("html_url"))

Both models were fine-tuned on the same real, self-collected dataset for the same number of
epochs, so the comparison table in Section 8 isolates the effect of model architecture. The
published release is what `PPOCR_License_Plate_Inference.ipynb` downloads next.